# Distil-BERT - Full fine tuning

In [1]:
import os
import random
import numpy as np
import pandas as pd
import torch

from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer

In [2]:
SEED = 42

MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 128

DATA_DIR = "data_audit_outputs"

TRAIN_PATH = os.path.join(DATA_DIR, "train.csv")
VAL_PATH = os.path.join(DATA_DIR, "validation.csv")
TEST_PATH = os.path.join(DATA_DIR, "test.csv")

OUTPUT_DIR = "distilbert_full_ft_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

label_id_to_name = {
    0: "Bearish",
    1: "Bullish",
    2: "Neutral"
}

label_name_to_id = {
    "Bearish": 0,
    "Bullish": 1,
    "Neutral": 2
}

In [3]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [4]:
train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)
test_df = pd.read_csv(TEST_PATH)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

In [5]:
train_df["label"] = train_df["label"].astype(int)
val_df["label"] = val_df["label"].astype(int)
test_df["label"] = test_df["label"].astype(int)

In [6]:
raw_datasets = DatasetDict({
    "train": Dataset.from_pandas(train_df, preserve_index=False),
    "validation": Dataset.from_pandas(val_df, preserve_index=False),
    "test": Dataset.from_pandas(test_df, preserve_index=False)
})

raw_datasets = raw_datasets.rename_column("label", "labels")

In [7]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [8]:
MAX_LENGTH = 128
def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH
    )
tokenized_datasets = raw_datasets.map(
    tokenize_batch,
    batched=True,
    remove_columns=["text", "label_name"]
)

tokenized_datasets.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

Map:   0%|          | 0/8105 [00:00<?, ? examples/s]

Map:   0%|          | 0/1431 [00:00<?, ? examples/s]

Map:   0%|          | 0/2388 [00:00<?, ? examples/s]

In [9]:
raw_dataframes = {
    "train": train_df,
    "validation": val_df,
    "test": test_df
}
sample = tokenized_datasets["train"][0]

print("input_ids shape:", sample["input_ids"].shape)
print("attention_mask shape:", sample["attention_mask"].shape)
print("label:", sample["labels"].item())

input_ids shape: torch.Size([128])
attention_mask shape: torch.Size([128])
label: 1


In [10]:
import time
import inspect

from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

LEARNING_RATE = 2e-5
NUM_EPOCHS = 4

TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32

WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06

LOGGING_STEPS = 50

DISTILBERT_RUN_NAME = "distilbert_full_ft"
DISTILBERT_OUTPUT_DIR = os.path.join(OUTPUT_DIR, "checkpoints")
os.makedirs(DISTILBERT_OUTPUT_DIR, exist_ok=True)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=label_id_to_name,
    label2id=label_name_to_id
)

model.to(device)

def count_parameters(model):
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total_params, trainable_params


total_params, trainable_params = count_parameters(model)

print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Trainable share:      {100 * trainable_params / total_params:.2f}%")

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Total parameters:     66,955,779
Trainable parameters: 66,955,779
Trainable share:      100.00%


In [11]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    preds = np.argmax(logits, axis=1)

    accuracy = accuracy_score(labels, preds)
    macro_f1 = f1_score(labels, preds, average="macro")
    weighted_f1 = f1_score(labels, preds, average="weighted")

    return {
        "accuracy": accuracy,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1
    }

training_args_kwargs = {
    "output_dir": DISTILBERT_OUTPUT_DIR,
    "learning_rate": LEARNING_RATE,
    "per_device_train_batch_size": TRAIN_BATCH_SIZE,
    "per_device_eval_batch_size": EVAL_BATCH_SIZE,
    "num_train_epochs": NUM_EPOCHS,
    "weight_decay": WEIGHT_DECAY,
    "warmup_ratio": WARMUP_RATIO,
    "logging_steps": LOGGING_STEPS,
    "save_strategy": "epoch",
    "load_best_model_at_end": True,
    "metric_for_best_model": "macro_f1",
    "greater_is_better": True,
    "save_total_limit": 2,
    "report_to": "none",
    "seed": SEED,
    "data_seed": SEED,
    "fp16": torch.cuda.is_available()
}

signature = inspect.signature(TrainingArguments.__init__)

if "eval_strategy" in signature.parameters:
    training_args_kwargs["eval_strategy"] = "epoch"
else:
    training_args_kwargs["evaluation_strategy"] = "epoch"

training_args = TrainingArguments(**training_args_kwargs)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [12]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=2)
    ]
)

start_time = time.time()

train_result = trainer.train()

end_time = time.time()
training_time_sec = end_time - start_time

print(f"Training time: {training_time_sec:.2f} sec")
print(f"Training time: {training_time_sec / 60:.2f} min")

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1
1,0.417180,0.395726,0.857442,0.807909,0.853933
2,0.324512,0.402821,0.860936,0.808812,0.856078
3,0.182798,0.423818,0.879804,0.843001,0.879783
4,0.080177,0.470695,0.875611,0.837391,0.875076


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


Training time: 179.04 sec
Training time: 2.98 min


In [13]:
validation_metrics = trainer.evaluate(tokenized_datasets["validation"])

validation_metrics

{'eval_loss': 0.4238061308860779,
 'eval_accuracy': 0.8798043326345213,
 'eval_macro_f1': 0.8426745416523187,
 'eval_weighted_f1': 0.8798060359115731,
 'eval_runtime': 1.267,
 'eval_samples_per_second': 1129.422,
 'eval_steps_per_second': 35.516,
 'epoch': 4.0}

In [14]:
validation_metrics_clean = {
    "model": DISTILBERT_RUN_NAME,
    "base_model": MODEL_NAME,
    "fine_tuning_type": "full_fine_tuning",
    "max_length": MAX_LENGTH,
    "learning_rate": LEARNING_RATE,
    "num_epochs": NUM_EPOCHS,
    "train_batch_size": TRAIN_BATCH_SIZE,
    "eval_batch_size": EVAL_BATCH_SIZE,
    "weight_decay": WEIGHT_DECAY,
    "warmup_ratio": WARMUP_RATIO,
    "total_params": total_params,
    "trainable_params": trainable_params,
    "trainable_share": trainable_params / total_params,
    "training_time_sec": training_time_sec,
    "best_checkpoint": trainer.state.best_model_checkpoint,
    "validation_accuracy": validation_metrics["eval_accuracy"],
    "validation_macro_f1": validation_metrics["eval_macro_f1"],
    "validation_weighted_f1": validation_metrics["eval_weighted_f1"],
    "validation_loss": validation_metrics["eval_loss"]
}

validation_metrics_df = pd.DataFrame([validation_metrics_clean])

validation_metrics_path = os.path.join(
    OUTPUT_DIR,
    "distilbert_full_ft_validation_metrics.csv"
)

validation_metrics_df.to_csv(validation_metrics_path, index=False)

validation_metrics_df

,model,base_model,fine_tuning_type,max_length,learning_rate,num_epochs,train_batch_size,eval_batch_size,weight_decay,warmup_ratio,total_params,trainable_params,trainable_share,training_time_sec,best_checkpoint,validation_accuracy,validation_macro_f1,validation_weighted_f1,validation_loss
0,distilbert_full_ft,distilbert-base-uncased,full_fine_tuning,128,0.00002,4,16,32,0.01,0.06,66955779,66955779,1.0,179.03968,distilbert_full_ft_outputs/checkpoints/checkpo...,0.879804,0.842675,0.879806,0.423806


In [15]:
BEST_MODEL_DIR = os.path.join(OUTPUT_DIR, "best_model")

trainer.save_model(BEST_MODEL_DIR)
tokenizer.save_pretrained(BEST_MODEL_DIR)

print("Best model saved to:", BEST_MODEL_DIR)
print("Best checkpoint:", trainer.state.best_model_checkpoint)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Best model saved to: distilbert_full_ft_outputs/best_model
Best checkpoint: distilbert_full_ft_outputs/checkpoints/checkpoint-1521


In [16]:
print("DistilBERT full fine-tuning finished")
print("-" * 50)
print(f"Base model:              {MODEL_NAME}")
print(f"Max length:              {MAX_LENGTH}")
print(f"Learning rate:           {LEARNING_RATE}")
print(f"Epochs:                  {NUM_EPOCHS}")
print(f"Train batch size:        {TRAIN_BATCH_SIZE}")
print(f"Eval batch size:         {EVAL_BATCH_SIZE}")
print(f"Total params:            {total_params:,}")
print(f"Trainable params:        {trainable_params:,}")
print(f"Training time, sec:      {training_time_sec:.2f}")
print(f"Best checkpoint:         {trainer.state.best_model_checkpoint}")
print()
print(f"Validation accuracy:     {validation_metrics['eval_accuracy']:.5f}")
print(f"Validation macro F1:     {validation_metrics['eval_macro_f1']:.5f}")
print(f"Validation weighted F1:  {validation_metrics['eval_weighted_f1']:.5f}")

DistilBERT full fine-tuning finished
--------------------------------------------------
Base model:              distilbert-base-uncased
Max length:              128
Learning rate:           2e-05
Epochs:                  4
Train batch size:        16
Eval batch size:         32
Total params:            66,955,779
Trainable params:        66,955,779
Training time, sec:      179.04
Best checkpoint:         distilbert_full_ft_outputs/checkpoints/checkpoint-1521

Validation accuracy:     0.87980
Validation macro F1:     0.84267
Validation weighted F1:  0.87981


In [17]:
def softmax_numpy(logits):
    logits = logits - np.max(logits, axis=1, keepdims=True)
    exp_logits = np.exp(logits)
    return exp_logits / np.sum(exp_logits, axis=1, keepdims=True)


def get_main_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted")
    }


def make_per_class_metrics_df(y_true, y_pred):
    target_names = [label_id_to_name[i] for i in range(3)]

    report = classification_report(
        y_true,
        y_pred,
        labels=[0, 1, 2],
        target_names=target_names,
        output_dict=True,
        zero_division=0
    )

    rows = []
    for class_id, class_name in label_id_to_name.items():
        rows.append({
            "class_id": class_id,
            "class_name": class_name,
            "precision": report[class_name]["precision"],
            "recall": report[class_name]["recall"],
            "f1_score": report[class_name]["f1-score"],
            "support": int(report[class_name]["support"])
        })

    return pd.DataFrame(rows)

In [18]:
validation_pred_output = trainer.predict(tokenized_datasets["validation"])

val_logits = validation_pred_output.predictions
val_probs = softmax_numpy(val_logits)
val_preds = np.argmax(val_probs, axis=1)

val_true = raw_dataframes["validation"]["label"].values

validation_main_metrics = get_main_metrics(val_true, val_preds)

validation_main_metrics

{'accuracy': 0.8798043326345213,
 'macro_f1': 0.8426745416523187,
 'weighted_f1': 0.8798060359115731}

In [19]:
test_pred_output = trainer.predict(tokenized_datasets["test"])

test_logits = test_pred_output.predictions
test_probs = softmax_numpy(test_logits)
test_preds = np.argmax(test_probs, axis=1)

test_true = raw_dataframes["test"]["label"].values

test_main_metrics = get_main_metrics(test_true, test_preds)

test_main_metrics

{'accuracy': 0.8819095477386935,
 'macro_f1': 0.8471207970860398,
 'weighted_f1': 0.8827361660252689}

In [20]:
validation_metrics_final = {
    "model": "distilbert_full_ft",
    "base_model": MODEL_NAME,
    "fine_tuning_type": "full_fine_tuning",
    "max_length": MAX_LENGTH,
    "total_params": total_params,
    "trainable_params": trainable_params,
    "trainable_share": trainable_params / total_params,
    "training_time_sec": training_time_sec,
    "best_checkpoint": trainer.state.best_model_checkpoint,
    "validation_accuracy": validation_main_metrics["accuracy"],
    "validation_macro_f1": validation_main_metrics["macro_f1"],
    "validation_weighted_f1": validation_main_metrics["weighted_f1"]
}

validation_metrics_final_df = pd.DataFrame([validation_metrics_final])

validation_metrics_final_path = os.path.join(
    OUTPUT_DIR,
    "distilbert_full_ft_validation_metrics.csv"
)

validation_metrics_final_df.to_csv(validation_metrics_final_path, index=False)

validation_metrics_final_df

,model,base_model,fine_tuning_type,max_length,total_params,trainable_params,trainable_share,training_time_sec,best_checkpoint,validation_accuracy,validation_macro_f1,validation_weighted_f1
0,distilbert_full_ft,distilbert-base-uncased,full_fine_tuning,128,66955779,66955779,1.0,179.03968,distilbert_full_ft_outputs/checkpoints/checkpo...,0.879804,0.842675,0.879806


In [21]:
test_metrics = {
    "model": "distilbert_full_ft",
    "base_model": MODEL_NAME,
    "fine_tuning_type": "full_fine_tuning",
    "max_length": MAX_LENGTH,
    "total_params": total_params,
    "trainable_params": trainable_params,
    "trainable_share": trainable_params / total_params,
    "training_time_sec": training_time_sec,
    "best_checkpoint": trainer.state.best_model_checkpoint,
    "test_accuracy": test_main_metrics["accuracy"],
    "test_macro_f1": test_main_metrics["macro_f1"],
    "test_weighted_f1": test_main_metrics["weighted_f1"]
}

test_metrics_df = pd.DataFrame([test_metrics])

test_metrics_path = os.path.join(
    OUTPUT_DIR,
    "distilbert_full_ft_test_metrics.csv"
)

test_metrics_df.to_csv(test_metrics_path, index=False)

test_metrics_df

,model,base_model,fine_tuning_type,max_length,total_params,trainable_params,trainable_share,training_time_sec,best_checkpoint,test_accuracy,test_macro_f1,test_weighted_f1
0,distilbert_full_ft,distilbert-base-uncased,full_fine_tuning,128,66955779,66955779,1.0,179.03968,distilbert_full_ft_outputs/checkpoints/checkpo...,0.88191,0.847121,0.882736


In [22]:
test_per_class_metrics_df = make_per_class_metrics_df(test_true, test_preds)

test_per_class_metrics_path = os.path.join(
    OUTPUT_DIR,
    "distilbert_full_ft_test_per_class_metrics.csv"
)

test_per_class_metrics_df.to_csv(test_per_class_metrics_path, index=False)

test_per_class_metrics_df

,class_id,class_name,precision,recall,f1_score,support
0,0,Bearish,0.761780,0.838617,0.798354,347
1,1,Bullish,0.826271,0.821053,0.823654,475
2,2,Neutral,0.928944,0.909962,0.919355,1566


In [23]:
class_names = [label_id_to_name[i] for i in range(3)]

test_cm = confusion_matrix(
    test_true,
    test_preds,
    labels=[0, 1, 2]
)

test_cm_df = pd.DataFrame(
    test_cm,
    index=[f"true_{name}" for name in class_names],
    columns=[f"pred_{name}" for name in class_names]
)

test_cm_path = os.path.join(
    OUTPUT_DIR,
    "distilbert_full_ft_test_confusion_matrix.csv"
)

test_cm_df.to_csv(test_cm_path)

test_cm_df

,pred_Bearish,pred_Bullish,pred_Neutral
true_Bearish,291,13,43
true_Bullish,19,390,66
true_Neutral,72,69,1425


In [24]:
test_predictions_df = raw_dataframes["test"].copy()

test_predictions_df["true_label"] = test_true
test_predictions_df["true_label_name"] = test_predictions_df["true_label"].map(label_id_to_name)

test_predictions_df["pred_label"] = test_preds
test_predictions_df["pred_label_name"] = test_predictions_df["pred_label"].map(label_id_to_name)

test_predictions_df["correct"] = (
    test_predictions_df["true_label"] == test_predictions_df["pred_label"]
)

test_predictions_df["prob_bearish"] = test_probs[:, 0]
test_predictions_df["prob_bullish"] = test_probs[:, 1]
test_predictions_df["prob_neutral"] = test_probs[:, 2]

test_predictions_path = os.path.join(
    OUTPUT_DIR,
    "distilbert_full_ft_test_predictions.csv"
)

test_predictions_df.to_csv(test_predictions_path, index=False)

test_predictions_df.head()

,text,label,label_name,true_label,true_label_name,pred_label,pred_label_name,correct,prob_bearish,prob_bullish,prob_neutral
0,$ALLY - Ally Financial pulls outlook https://t...,0,Bearish,0,Bearish,0,Bearish,True,0.982772,0.008265,0.008963
1,"$DELL $HPE - Dell, HPE targets trimmed on comp...",0,Bearish,0,Bearish,0,Bearish,True,0.980777,0.009077,0.010146
2,$PRTY - Moody's turns negative on Party City h...,0,Bearish,0,Bearish,0,Bearish,True,0.986282,0.005986,0.007732
3,$SAN: Deutsche Bank cuts to Hold,0,Bearish,0,Bearish,0,Bearish,True,0.986720,0.005884,0.007395
4,$SITC: Compass Point cuts to Sell,0,Bearish,0,Bearish,0,Bearish,True,0.987296,0.005629,0.007074


In [25]:
final_results_for_main_table = {
    "model": "DistilBERT",
    "experiment_name": "distilbert_full_ft",
    "base_model": MODEL_NAME,
    "fine_tuning_type": "full_fine_tuning",
    "max_length": MAX_LENGTH,
    "total_params": total_params,
    "trainable_params": trainable_params,
    "trainable_share": trainable_params / total_params,
    "training_time_sec": training_time_sec,
    "best_checkpoint": trainer.state.best_model_checkpoint,

    "validation_accuracy": validation_main_metrics["accuracy"],
    "validation_macro_f1": validation_main_metrics["macro_f1"],
    "validation_weighted_f1": validation_main_metrics["weighted_f1"],

    "test_accuracy": test_main_metrics["accuracy"],
    "test_macro_f1": test_main_metrics["macro_f1"],
    "test_weighted_f1": test_main_metrics["weighted_f1"]
}

final_results_for_main_table_df = pd.DataFrame([final_results_for_main_table])

final_results_for_main_table_path = os.path.join(
    OUTPUT_DIR,
    "distilbert_full_ft_final_results_for_main_table.csv"
)

final_results_for_main_table_df.to_csv(
    final_results_for_main_table_path,
    index=False
)

final_results_for_main_table_df

,model,experiment_name,base_model,fine_tuning_type,max_length,total_params,trainable_params,trainable_share,training_time_sec,best_checkpoint,validation_accuracy,validation_macro_f1,validation_weighted_f1,test_accuracy,test_macro_f1,test_weighted_f1
0,DistilBERT,distilbert_full_ft,distilbert-base-uncased,full_fine_tuning,128,66955779,66955779,1.0,179.03968,distilbert_full_ft_outputs/checkpoints/checkpo...,0.879804,0.842675,0.879806,0.88191,0.847121,0.882736


In [26]:
import shutil
from google.colab import files

folder_name = "distilbert_full_ft_outputs"
zip_name = "distilbert_full_ft_outputs"

shutil.make_archive(zip_name, "zip", folder_name)

files.download(zip_name + ".zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>